# Cosmological Parameter Inference with Pantheon+SH0ES
**Model**: Flat ΛCDM with the Tripp (1998) distance estimator (Eq. 1 of Brout et al. 2022)

$$\mu_{\rm obs} = m_B + \alpha x_1 - \beta c - M - \delta_{\rm bias} + \delta_{\rm host}$$

We set $\delta_{\rm bias} = \delta_{\rm host} = 0$ and infer $(H_0, \Omega_m, \alpha, \beta, M)$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import emcee
import corner
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u

## 1. Load data

In [ ]:
df = pd.read_csv('data/Pantheon+SH0ES.dat', sep='\s+', comment='#')

# Exclude Cepheid calibrators (very low-z, absolute distance anchors)
# and keep only Hubble-flow SNe
mask = (df['zHD'] > 0.01) & (df['IS_CALIBRATOR'] == 0)
df   = df[mask].reset_index(drop=True)

# Observables from SALT2 fit (Tripp estimator inputs)
z    = df['zHD'].values          # Hubble-diagram redshift
mB   = df['mB'].values           # uncorrected peak magnitude
x1   = df['x1'].values           # stretch
c    = df['c'].values            # color
mB_err = df['mBERR'].values      # uncertainty on mB
x1_err = df['x1ERR'].values
c_err  = df['cERR'].values

# For plotting only (diagonal errors, not used in the likelihood fit)
mu_plot     = df['MU_SH0ES'].values
mu_plot_err = df['MU_SH0ES_ERR_DIAG'].values

print(f'SNe in fit: {len(z)}')
print(f'Redshift range: {z.min():.4f} – {z.max():.4f}')

## 2. Model

**Observed distance modulus** (Tripp estimator, $\delta$s = 0):
$$\mu_{\rm obs}(\alpha,\beta,M) = m_B + \alpha x_1 - \beta c - M$$

**Theoretical distance modulus** (flat ΛCDM):
$$\mu_{\rm th}(z, H_0, \Omega_m) = 5\log_{10}\left(d_L(z)/\text{Mpc}\right) + 25$$

**Statistical uncertainty** on $\mu_{\rm obs}$ (error propagation from SALT2):
$$\sigma_{\mu}^2 = \sigma_{m_B}^2 + \alpha^2\sigma_{x_1}^2 + \beta^2\sigma_c^2$$

(Cross-terms from the full covariance matrix are ignored here for simplicity.)

In [ ]:
def mu_theory(z, H0, Om0):
    """Theoretical distance modulus for flat ΛCDM via astropy."""
    cosmo = FlatLambdaCDM(H0=H0, Om0=Om0)
    dL    = cosmo.luminosity_distance(z).to(u.Mpc).value
    return 5.0 * np.log10(dL) + 25.0

def mu_obs(mB, x1, c, alpha, beta, M):
    """Observed distance modulus from Tripp estimator (Eq. 1, deltas=0)."""
    delta_bias = 0.0   # set to 0; could use df['BIAS_...'] column if available
    delta_host = 0.0   # set to 0; could use host-mass step function (Eq. 2)
    return mB + alpha * x1 - beta * c - M - delta_bias + delta_host

def sigma_mu(alpha, beta, mB_err, x1_err, c_err):
    """Statistical uncertainty on mu_obs from error propagation (diagonal only)."""
    return np.sqrt(mB_err**2 + (alpha * x1_err)**2 + (beta * c_err)**2)

## 3. Sanity check — model curves vs data

In [ ]:
# Typical SALT2 nuisance values from the literature
alpha0, beta0, M0 = 0.15, 3.1, -19.3

mu_obs_check = mu_obs(mB, x1, c, alpha0, beta0, M0)
sig_check    = sigma_mu(alpha0, beta0, mB_err, x1_err, c_err)

z_th = np.geomspace(z.min(), z.max(), 500)

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(z, mu_obs_check, yerr=sig_check, fmt='.', color='gray',
            alpha=0.2, ms=2, elinewidth=0.4, capsize=0, label='Data (Tripp μ)')
for H0, Om0, lbl, col in [
    (70, 0.3,  r'$H_0=70,\,\Omega_m=0.3$',  'C0'),
    (73, 0.3,  r'$H_0=73,\,\Omega_m=0.3$',  'C1'),
    (70, 0.15, r'$H_0=70,\,\Omega_m=0.15$', 'C2'),
]:
    ax.plot(z_th, mu_theory(z_th, H0, Om0), lw=2, color=col, label=lbl)
ax.set_xscale('log')
ax.set_xlabel('Redshift $z$'); ax.set_ylabel(r'$\mu$')
ax.set_title(r'Sanity check: Tripp $\mu_{\rm obs}$ vs flat $\Lambda$CDM')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 4. Log-posterior

Parameters: $\theta = (H_0,\, \Omega_m,\, \alpha,\, \beta,\, M)$

$$\ln\mathcal{L} = -\frac{1}{2}\sum_i \frac{[\mu_{\rm obs,i} - \mu_{\rm th}(z_i)]^2}{\sigma_{\mu,i}^2} + \ln(2\pi\sigma_{\mu,i}^2)$$

Flat priors: $H_0\in(50,100)$, $\Omega_m\in(0.05,0.7)$, $\alpha\in(0,1)$, $\beta\in(1,5)$, $M\in(-20,-18)$.

In [ ]:
# Prior bounds — physically motivated ranges
BOUNDS = {
    'H0':    (50,   100),
    'Om0':   (0.05, 0.70),
    'alpha': (0.0,  1.0),
    'beta':  (1.0,  5.0),
    'M':     (-20.5, -18.0),
}

def log_prior(theta):
    H0, Om0, alpha, beta, M = theta
    for val, (lo, hi) in zip(theta, BOUNDS.values()):
        if not (lo < val < hi):
            return -np.inf
    return 0.0   # flat prior inside bounds

def log_likelihood(theta, z, mB, x1, c, mB_err, x1_err, c_err):
    H0, Om0, alpha, beta, M = theta
    mu_o   = mu_obs(mB, x1, c, alpha, beta, M)
    mu_t   = mu_theory(z, H0, Om0)
    sig2   = sigma_mu(alpha, beta, mB_err, x1_err, c_err)**2
    resid  = mu_o - mu_t
    return -0.5 * np.sum(resid**2 / sig2 + np.log(2 * np.pi * sig2))

def log_probability(theta, z, mB, x1, c, mB_err, x1_err, c_err):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, z, mB, x1, c, mB_err, x1_err, c_err)

# Quick evaluation at fiducial values
theta0 = np.array([70.0, 0.3, 0.15, 3.1, -19.3])
print(f'log P at fiducial: {log_probability(theta0, z, mB, x1, c, mB_err, x1_err, c_err):.2f}')

## 5. Run emcee

In [ ]:
NWALKERS = 32
NDIM     = 5
NSTEPS   = 3000

# Start walkers in a small Gaussian ball around the fiducial point
np.random.seed(42)
pos = theta0 + 1e-3 * np.random.randn(NWALKERS, NDIM)

data_args = (z, mB, x1, c, mB_err, x1_err, c_err)

sampler = emcee.EnsembleSampler(NWALKERS, NDIM, log_probability, args=data_args)
sampler.run_mcmc(pos, NSTEPS, progress=True)

## 6. Convergence — trace plots & autocorrelation

In [ ]:
LABELS = [r'$H_0$', r'$\Omega_m$', r'$\alpha$', r'$\beta$', r'$M$']

fig, axes = plt.subplots(NDIM, figsize=(11, 8), sharex=True)
raw = sampler.get_chain()
for i, (ax, lbl) in enumerate(zip(axes, LABELS)):
    ax.plot(raw[:, :, i], 'k', alpha=0.2, lw=0.4)
    ax.set_ylabel(lbl, fontsize=10)
axes[-1].set_xlabel('Step')
fig.suptitle('Trace plots (all walkers)', fontsize=12)
plt.tight_layout(); plt.show()

# Autocorrelation time → burn-in and thinning
try:
    tau    = sampler.get_autocorr_time(quiet=True)
    tau    = np.nan_to_num(tau, nan=100.0)
    burnin = int(3 * tau.max())
    thin   = max(1, int(tau.max() / 2))
    print(f'τ = {tau.round(1)}')
except Exception:
    burnin, thin = 500, 15
print(f'burn-in={burnin}  thin={thin}')

## 7. Flat samples & percentile results

In [ ]:
flat = sampler.get_chain(discard=burnin, thin=thin, flat=True)
print(f'Flat samples: {flat.shape}\n')

PARAM_NAMES = ['H0', 'Om0', 'alpha', 'beta', 'M']
print(f'{"Param":>8}  {"16%":>9}  {"50% (med)":>11}  {"84%":>9}  {"-σ":>7}  {"+σ":>7}')
print('-' * 60)
for i, (lbl, name) in enumerate(zip(LABELS, PARAM_NAMES)):
    p16, p50, p84 = np.percentile(flat[:, i], [16, 50, 84])
    print(f'{name:>8}  {p16:>9.4f}  {p50:>11.4f}  {p84:>9.4f}  {p50-p16:>7.4f}  {p84-p50:>7.4f}')

## 8. Corner plot

In [ ]:
fig = corner.corner(
    flat, labels=LABELS,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True, title_fmt='.3f',
    plot_datapoints=False, smooth=1.0,
)
fig.suptitle('Posterior — flat ΛCDM + Tripp nuisance params', y=1.01, fontsize=13)
plt.savefig('corner_pantheon.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Posterior predictive — Hubble diagram

In [ ]:
med = np.median(flat, axis=0)
H0_med, Om0_med, a_med, b_med, M_med = med

z_th   = np.geomspace(z.min(), z.max(), 500)
mu_med = mu_theory(z_th, H0_med, Om0_med)

# Compute mu_obs with median nuisance params for plotting
mu_o_plot = mu_obs(mB, x1, c, a_med, b_med, M_med)
sig_plot  = sigma_mu(a_med, b_med, mB_err, x1_err, c_err)

fig, axes = plt.subplots(2, 1, figsize=(10, 8),
                          gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
ax, ax_res = axes

# Posterior predictive band (100 random samples)
idx = np.random.choice(len(flat), 150, replace=False)
for s in flat[idx]:
    ax.plot(z_th, mu_theory(z_th, s[0], s[1]), color='C0', alpha=0.04, lw=0.8)

ax.errorbar(z, mu_o_plot, yerr=sig_plot, fmt='.', color='gray',
            alpha=0.25, ms=2, elinewidth=0.4, capsize=0, label='Data')
ax.plot(z_th, mu_med, 'C0', lw=2,
        label=f'Median: $H_0$={H0_med:.1f}, $\\Omega_m$={Om0_med:.3f}')
ax.set_ylabel(r'$\mu$', fontsize=12)
ax.set_title('Posterior predictive Hubble diagram', fontsize=12)
ax.legend(fontsize=9); ax.set_xscale('log')

# Residuals
ax_res.errorbar(z, mu_o_plot - mu_theory(z, H0_med, Om0_med),
                yerr=sig_plot, fmt='.', color='gray', alpha=0.25,
                ms=2, elinewidth=0.4, capsize=0)
ax_res.axhline(0, color='C0', lw=1.5)
ax_res.set_ylabel(r'$\Delta\mu$', fontsize=11)
ax_res.set_xlabel('Redshift $z$', fontsize=12)
ax_res.set_ylim(-0.8, 0.8); ax_res.set_xscale('log')

plt.tight_layout()
plt.savefig('hubble_posterior.png', dpi=130, bbox_inches='tight')
plt.show()